# 03 — Clinical Trials Evidence Exploration

**Stage 4 of the pipeline:** add **clinical trial evidence** for each EGFR drug.

Flow: load drug recommendations (notebook 01) → search ClinicalTrials.gov (API v2) per drug → extract trial id/title/status/phase/conditions/interventions → save evidence + per-drug summary CSVs.

Outputs: `egfr_clinical_trials.csv`, `egfr_clinical_trials_summary.csv`

### 1. Test notebook environment

In [1]:
import sys
import time
from pathlib import Path

import requests
import pandas as pd

print("Notebook is working")
print("Python executable:", sys.executable)

Notebook is working
Python executable: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/.venv/bin/python


### 2. Set project folders

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

print("Project root:", PROJECT_ROOT)
print("Processed data folder:", PROCESSED_DIR)

Project root: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant
Processed data folder: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed


### 3. Load EGFR drug recommendations (from notebook 01)

In [3]:
recommendations_file = PROCESSED_DIR / "egfr_drug_recommendations.csv"

if not recommendations_file.exists():
    raise FileNotFoundError(
        "egfr_drug_recommendations.csv not found. Run Notebook 01 first."
    )

drug_recommendations_df = pd.read_csv(recommendations_file)
print("Rows:", len(drug_recommendations_df))
drug_recommendations_df.head()

Rows: 76


,drug_name,molecule_chembl_id,action_type,mechanism_of_action,approval_status,max_phase,target_name
0,PANITUMUMAB,CHEMBL1201827,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Approved,4.0,Epidermal growth factor receptor
1,CETUXIMAB,CHEMBL1201577,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Approved,4.0,Epidermal growth factor receptor
2,ERLOTINIB HYDROCHLORIDE,CHEMBL1079742,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Approved,4.0,Epidermal growth factor receptor
3,GEFITINIB,CHEMBL939,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Approved,4.0,Epidermal growth factor receptor
4,LAPATINIB DITOSYLATE,CHEMBL1201179,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Approved,4.0,Epidermal growth factor receptor


### 4. Select drugs to search
Start with the first 5; remove `.head(5)` later to search all.

In [4]:
target_name = "EGFR"

if "drug_name" not in drug_recommendations_df.columns:
    raise ValueError("drug_name column not found in egfr_drug_recommendations.csv")

drugs_to_search = (
    drug_recommendations_df["drug_name"]
    .dropna()
    .drop_duplicates()
    .head(5)
    .tolist()
)

print("Target:", target_name)
print("Drugs selected for ClinicalTrials.gov search:")
for drug in drugs_to_search:
    print("-", drug)

Target: EGFR
Drugs selected for ClinicalTrials.gov search:
- PANITUMUMAB
- CETUXIMAB
- ERLOTINIB HYDROCHLORIDE
- GEFITINIB
- LAPATINIB DITOSYLATE


### 5. ClinicalTrials.gov API helper
Note: `countTotal=true` is required for the API to return `totalCount`.

In [5]:
CLINICAL_TRIALS_URL = "https://clinicaltrials.gov/api/v2/studies"


def clinical_trials_get(params, retries=3, pause=1):
    """GET JSON from ClinicalTrials.gov API v2 with simple retries."""
    for attempt in range(retries):
        try:
            response = requests.get(CLINICAL_TRIALS_URL, params=params, timeout=(10, 60))
            if response.status_code == 200:
                return response.json()
            print(f"  attempt {attempt + 1}: HTTP {response.status_code}, retrying...")
        except requests.exceptions.RequestException as error:
            print(f"  attempt {attempt + 1}: {type(error).__name__}, retrying...")
        time.sleep(pause)
    return None


def build_clinical_trial_query(drug_name, target_name="EGFR"):
    """Simple search query, e.g. 'ERLOTINIB EGFR'."""
    return f"{drug_name} {target_name}"

### 6. Quick test for one drug

In [6]:
test_drug = drugs_to_search[0]
test_query = build_clinical_trial_query(test_drug, target_name)

params = {"format": "json", "query.term": test_query, "pageSize": 5, "countTotal": "true"}
test_data = clinical_trials_get(params)

if test_data is None:
    raise RuntimeError("ClinicalTrials.gov test search failed.")

print("Test drug:", test_drug)
print("Query:", test_query)
print("Total count:", test_data.get("totalCount"))
print("Studies returned:", len(test_data.get("studies", [])))

Test drug: PANITUMUMAB
Query: PANITUMUMAB EGFR
Total count: 146
Studies returned: 5


### 7. Helpers to safely extract trial fields

In [7]:
def safe_join(value):
    """Convert list values into readable text."""
    if isinstance(value, list):
        return " | ".join([str(item) for item in value])
    if value is None:
        return None
    return str(value)


def extract_interventions(arms_interventions_module):
    """Extract intervention names from a trial record."""
    interventions = arms_interventions_module.get("interventions", [])
    names = []
    for intervention in interventions:
        name = intervention.get("name")
        itype = intervention.get("type")
        if name and itype:
            names.append(f"{name} ({itype})")
        elif name:
            names.append(name)
    return " | ".join(names)


def extract_trial_record(study, drug_name, query, target_name="EGFR"):
    """Extract useful fields from one ClinicalTrials.gov study record."""
    protocol = study.get("protocolSection", {})
    identification = protocol.get("identificationModule", {})
    status = protocol.get("statusModule", {})
    conditions = protocol.get("conditionsModule", {})
    design = protocol.get("designModule", {})
    arms = protocol.get("armsInterventionsModule", {})
    description = protocol.get("descriptionModule", {})

    nct_id = identification.get("nctId")
    return {
        "target_name": target_name,
        "drug_name": drug_name,
        "query": query,
        "nct_id": nct_id,
        "brief_title": identification.get("briefTitle"),
        "overall_status": status.get("overallStatus"),
        "start_date": status.get("startDateStruct", {}).get("date"),
        "completion_date": status.get("completionDateStruct", {}).get("date"),
        "study_type": design.get("studyType"),
        "phases": safe_join(design.get("phases")),
        "conditions": safe_join(conditions.get("conditions")),
        "interventions": extract_interventions(arms),
        "url": f"https://clinicaltrials.gov/study/{nct_id}" if nct_id else None,
        "source": "ClinicalTrials.gov",
    }

### 8. Search ClinicalTrials.gov for all selected drugs

In [8]:
clinical_trial_records = []

for drug in drugs_to_search:
    query = build_clinical_trial_query(drug, target_name)
    params = {"format": "json", "query.term": query, "pageSize": 10, "countTotal": "true"}

    data = clinical_trials_get(params)
    if data is None:
        print(f"Search failed for: {drug}")
        continue

    total_count = data.get("totalCount", 0)
    studies = data.get("studies", [])
    print(f"{drug}: total matches = {total_count}, fetched = {len(studies)}")

    for study in studies:
        record = extract_trial_record(study, drug, query, target_name)
        record["total_matches_for_query"] = total_count
        clinical_trial_records.append(record)
    time.sleep(0.5)

clinical_trials_df = pd.DataFrame(clinical_trial_records)
print("Total clinical trial records collected:", len(clinical_trials_df))
clinical_trials_df.head(10)

PANITUMUMAB: total matches = 146, fetched = 10
CETUXIMAB: total matches = 400, fetched = 10
ERLOTINIB HYDROCHLORIDE: total matches = 464, fetched = 10
GEFITINIB: total matches = 273, fetched = 10
LAPATINIB DITOSYLATE: total matches = 120, fetched = 10
Total clinical trial records collected: 50


,target_name,drug_name,query,nct_id,brief_title,overall_status,start_date,completion_date,study_type,phases,conditions,interventions,url,source,total_matches_for_query
0,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT03983993,Niraparib and Panitumumab in Patients With Adv...,ACTIVE_NOT_RECRUITING,2019-10-15,2026-12-01,INTERVENTIONAL,PHASE2,Advanced Microsatellite Stable Colorectal Carc...,Niraparib (DRUG) | Panitumumab (BIOLOGICAL),https://clinicaltrials.gov/study/NCT03983993,ClinicalTrials.gov,146
1,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT03146338,Prophylaxis of Magnesium-rich Mineral Water to...,UNKNOWN,2017-07-04,2023-01-04,INTERVENTIONAL,NA,Metastatic Colorectal Cancer | Metastatic Head...,Magnesium-rich mineral water (Rozana) (OTHER),https://clinicaltrials.gov/study/NCT03146338,ClinicalTrials.gov,146
2,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT00346099,Study of Panitumumab Given First With Capecita...,WITHDRAWN,2006-06,2007-05,INTERVENTIONAL,PHASE2,Rectal Cancer | Neoplasm Metastasis,Panitumumab with capecitabine and oxaliplatin ...,https://clinicaltrials.gov/study/NCT00346099,ClinicalTrials.gov,146
3,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT01668498,Comparison of Two Preemptive Treatment Strateg...,COMPLETED,2011-05,2016-03,INTERVENTIONAL,PHASE2,Ras-wildtype Colorectal Cancer,Erythromycin (DRUG) | Doxycycline (DRUG),https://clinicaltrials.gov/study/NCT01668498,ClinicalTrials.gov,146
4,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT04034173,Optimal Anti-EGFR Treatment of mCRC Patients W...,NOT_YET_RECRUITING,2019-08-01,2026-08-01,INTERVENTIONAL,PHASE2,Treatment Related Cancer,Panitumumab (DRUG) | Irinotecan (DRUG) | Folin...,https://clinicaltrials.gov/study/NCT04034173,ClinicalTrials.gov,146
5,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT04787341,PAnitumumab REchallenge Followed by REgorafeni...,ACTIVE_NOT_RECRUITING,2020-12-15,2026-06-30,INTERVENTIONAL,PHASE2,Colorectal Cancer,Regorafenib (DRUG) | Panitumumab (DRUG),https://clinicaltrials.gov/study/NCT04787341,ClinicalTrials.gov,146
6,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT01380262,Pre-emptive Low-dose Doxycycline During Anti-E...,UNKNOWN,2010-06,2011-09,OBSERVATIONAL,NaN,Colorectal Cancer | Skin Toxicities,,https://clinicaltrials.gov/study/NCT01380262,ClinicalTrials.gov,146
7,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT03986541,"AREG, EREG and EGFR: Response to Anti-EGFR Age...",COMPLETED,2019-09-23,2022-07-22,OBSERVATIONAL,NaN,Colorectal Cancer Stage IV,Immunohistochemistry (DIAGNOSTIC_TEST),https://clinicaltrials.gov/study/NCT03986541,ClinicalTrials.gov,146
8,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT00115765,PACCE: Panitumumab Advanced Colorectal Cancer ...,COMPLETED,2005-06-01,2009-05-01,INTERVENTIONAL,PHASE3,Colorectal Cancer,Oxaliplatin Based Chemotherapy (DRUG) | Panitu...,https://clinicaltrials.gov/study/NCT00115765,ClinicalTrials.gov,146
9,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT01393821,Menadione Topical Lotion in Treating Skin Disc...,COMPLETED,2012-01-23,2018-07-14,INTERVENTIONAL,NA,Dermatologic Complications | Malignant Neoplas...,menadione topical lotion (DRUG) | placebo (OTH...,https://clinicaltrials.gov/study/NCT01393821,ClinicalTrials.gov,146


### 9. Remove duplicate trials (same drug + NCT id)

In [9]:
if clinical_trials_df.empty:
    clean_clinical_trials_df = pd.DataFrame(
        columns=["target_name", "drug_name", "nct_id", "brief_title", "overall_status",
                 "phases", "conditions", "interventions", "url", "source"]
    )
else:
    clean_clinical_trials_df = clinical_trials_df.drop_duplicates(
        subset=["drug_name", "nct_id"]
    ).copy()

print("Rows before cleaning:", len(clinical_trials_df))
print("Rows after cleaning:", len(clean_clinical_trials_df))
clean_clinical_trials_df.head(10)

Rows before cleaning: 50
Rows after cleaning: 50


,target_name,drug_name,query,nct_id,brief_title,overall_status,start_date,completion_date,study_type,phases,conditions,interventions,url,source,total_matches_for_query
0,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT03983993,Niraparib and Panitumumab in Patients With Adv...,ACTIVE_NOT_RECRUITING,2019-10-15,2026-12-01,INTERVENTIONAL,PHASE2,Advanced Microsatellite Stable Colorectal Carc...,Niraparib (DRUG) | Panitumumab (BIOLOGICAL),https://clinicaltrials.gov/study/NCT03983993,ClinicalTrials.gov,146
1,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT03146338,Prophylaxis of Magnesium-rich Mineral Water to...,UNKNOWN,2017-07-04,2023-01-04,INTERVENTIONAL,NA,Metastatic Colorectal Cancer | Metastatic Head...,Magnesium-rich mineral water (Rozana) (OTHER),https://clinicaltrials.gov/study/NCT03146338,ClinicalTrials.gov,146
2,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT00346099,Study of Panitumumab Given First With Capecita...,WITHDRAWN,2006-06,2007-05,INTERVENTIONAL,PHASE2,Rectal Cancer | Neoplasm Metastasis,Panitumumab with capecitabine and oxaliplatin ...,https://clinicaltrials.gov/study/NCT00346099,ClinicalTrials.gov,146
3,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT01668498,Comparison of Two Preemptive Treatment Strateg...,COMPLETED,2011-05,2016-03,INTERVENTIONAL,PHASE2,Ras-wildtype Colorectal Cancer,Erythromycin (DRUG) | Doxycycline (DRUG),https://clinicaltrials.gov/study/NCT01668498,ClinicalTrials.gov,146
4,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT04034173,Optimal Anti-EGFR Treatment of mCRC Patients W...,NOT_YET_RECRUITING,2019-08-01,2026-08-01,INTERVENTIONAL,PHASE2,Treatment Related Cancer,Panitumumab (DRUG) | Irinotecan (DRUG) | Folin...,https://clinicaltrials.gov/study/NCT04034173,ClinicalTrials.gov,146
5,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT04787341,PAnitumumab REchallenge Followed by REgorafeni...,ACTIVE_NOT_RECRUITING,2020-12-15,2026-06-30,INTERVENTIONAL,PHASE2,Colorectal Cancer,Regorafenib (DRUG) | Panitumumab (DRUG),https://clinicaltrials.gov/study/NCT04787341,ClinicalTrials.gov,146
6,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT01380262,Pre-emptive Low-dose Doxycycline During Anti-E...,UNKNOWN,2010-06,2011-09,OBSERVATIONAL,NaN,Colorectal Cancer | Skin Toxicities,,https://clinicaltrials.gov/study/NCT01380262,ClinicalTrials.gov,146
7,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT03986541,"AREG, EREG and EGFR: Response to Anti-EGFR Age...",COMPLETED,2019-09-23,2022-07-22,OBSERVATIONAL,NaN,Colorectal Cancer Stage IV,Immunohistochemistry (DIAGNOSTIC_TEST),https://clinicaltrials.gov/study/NCT03986541,ClinicalTrials.gov,146
8,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT00115765,PACCE: Panitumumab Advanced Colorectal Cancer ...,COMPLETED,2005-06-01,2009-05-01,INTERVENTIONAL,PHASE3,Colorectal Cancer,Oxaliplatin Based Chemotherapy (DRUG) | Panitu...,https://clinicaltrials.gov/study/NCT00115765,ClinicalTrials.gov,146
9,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT01393821,Menadione Topical Lotion in Treating Skin Disc...,COMPLETED,2012-01-23,2018-07-14,INTERVENTIONAL,NA,Dermatologic Complications | Malignant Neoplas...,menadione topical lotion (DRUG) | placebo (OTH...,https://clinicaltrials.gov/study/NCT01393821,ClinicalTrials.gov,146


### 10. Save clinical trials evidence dataset

In [10]:
clinical_trials_file = PROCESSED_DIR / "egfr_clinical_trials.csv"
clean_clinical_trials_df.to_csv(clinical_trials_file, index=False)
print("Saved:", clinical_trials_file)

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/egfr_clinical_trials.csv


### 11. Per-drug clinical trial summary

In [11]:
active_statuses = [
    "RECRUITING", "NOT_YET_RECRUITING", "ACTIVE_NOT_RECRUITING", "ENROLLING_BY_INVITATION",
]


def count_active_trials(status_series):
    return status_series.isin(active_statuses).sum()


def count_late_phase_trials(phase_series):
    count = 0
    for phase_text in phase_series.dropna():
        t = str(phase_text).upper()
        if "PHASE3" in t or "PHASE 3" in t or "PHASE4" in t or "PHASE 4" in t:
            count += 1
    return count


if clean_clinical_trials_df.empty:
    clinical_trials_summary_df = pd.DataFrame(
        columns=["target_name", "drug_name", "clinical_trial_count",
                 "active_trial_count", "late_phase_trial_count", "top_trial_titles"]
    )
else:
    clinical_trials_summary_df = (
        clean_clinical_trials_df.groupby(["target_name", "drug_name"])
        .agg(
            clinical_trial_count=("nct_id", "nunique"),
            active_trial_count=("overall_status", count_active_trials),
            late_phase_trial_count=("phases", count_late_phase_trials),
            top_trial_titles=("brief_title", lambda s: " | ".join(list(s.dropna())[:3])),
        )
        .reset_index()
    )

clinical_trials_summary_df

,target_name,drug_name,clinical_trial_count,active_trial_count,late_phase_trial_count,top_trial_titles
0,EGFR,CETUXIMAB,10,1,2,A Phase II Trial of Modified FOLFOX 6 and Cetu...
1,EGFR,ERLOTINIB HYDROCHLORIDE,10,0,1,ARQ 197 Plus Erlotinib in Patient With Locally...
2,EGFR,GEFITINIB,10,0,2,ARQ 197 Plus Erlotinib in Patient With Locally...
3,EGFR,LAPATINIB DITOSYLATE,10,0,3,T-DM1 With Abraxane and Lapatinib for Metastat...
4,EGFR,PANITUMUMAB,10,3,1,Niraparib and Panitumumab in Patients With Adv...


### 12. Add a simple clinical-evidence score

In [12]:
def calculate_clinical_evidence_score(row):
    score = 0.0
    if row["clinical_trial_count"] > 0:
        score += 0.4
    if row["active_trial_count"] > 0:
        score += 0.2
    if row["late_phase_trial_count"] > 0:
        score += 0.3
    if row["clinical_trial_count"] >= 5:
        score += 0.1
    return round(min(score, 1.0), 2)


if not clinical_trials_summary_df.empty:
    clinical_trials_summary_df["clinical_evidence_score"] = clinical_trials_summary_df.apply(
        calculate_clinical_evidence_score, axis=1
    )
else:
    clinical_trials_summary_df["clinical_evidence_score"] = []

clinical_trials_summary_df

,target_name,drug_name,clinical_trial_count,active_trial_count,late_phase_trial_count,top_trial_titles,clinical_evidence_score
0,EGFR,CETUXIMAB,10,1,2,A Phase II Trial of Modified FOLFOX 6 and Cetu...,1.0
1,EGFR,ERLOTINIB HYDROCHLORIDE,10,0,1,ARQ 197 Plus Erlotinib in Patient With Locally...,0.8
2,EGFR,GEFITINIB,10,0,2,ARQ 197 Plus Erlotinib in Patient With Locally...,0.8
3,EGFR,LAPATINIB DITOSYLATE,10,0,3,T-DM1 With Abraxane and Lapatinib for Metastat...,0.8
4,EGFR,PANITUMUMAB,10,3,1,Niraparib and Panitumumab in Patients With Adv...,1.0


### 13. Save clinical trials summary dataset

In [13]:
clinical_trials_summary_file = PROCESSED_DIR / "egfr_clinical_trials_summary.csv"
clinical_trials_summary_df.to_csv(clinical_trials_summary_file, index=False)
print("Saved:", clinical_trials_summary_file)

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/egfr_clinical_trials_summary.csv


### 14. Final result

In [14]:
print("Clinical Trials Evidence Exploration Complete")
print("=" * 70)
print("Target:", target_name)
print("Drugs searched:", len(drugs_to_search))
print("Clinical trial evidence records:", len(clean_clinical_trials_df))
print("Clinical trial summary rows:", len(clinical_trials_summary_df))
display(clinical_trials_summary_df)
display(clean_clinical_trials_df.head(10))

Clinical Trials Evidence Exploration Complete
Target: EGFR
Drugs searched: 5
Clinical trial evidence records: 50
Clinical trial summary rows: 5


,target_name,drug_name,clinical_trial_count,active_trial_count,late_phase_trial_count,top_trial_titles,clinical_evidence_score
0,EGFR,CETUXIMAB,10,1,2,A Phase II Trial of Modified FOLFOX 6 and Cetu...,1.0
1,EGFR,ERLOTINIB HYDROCHLORIDE,10,0,1,ARQ 197 Plus Erlotinib in Patient With Locally...,0.8
2,EGFR,GEFITINIB,10,0,2,ARQ 197 Plus Erlotinib in Patient With Locally...,0.8
3,EGFR,LAPATINIB DITOSYLATE,10,0,3,T-DM1 With Abraxane and Lapatinib for Metastat...,0.8
4,EGFR,PANITUMUMAB,10,3,1,Niraparib and Panitumumab in Patients With Adv...,1.0


,target_name,drug_name,query,nct_id,brief_title,overall_status,start_date,completion_date,study_type,phases,conditions,interventions,url,source,total_matches_for_query
0,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT03983993,Niraparib and Panitumumab in Patients With Adv...,ACTIVE_NOT_RECRUITING,2019-10-15,2026-12-01,INTERVENTIONAL,PHASE2,Advanced Microsatellite Stable Colorectal Carc...,Niraparib (DRUG) | Panitumumab (BIOLOGICAL),https://clinicaltrials.gov/study/NCT03983993,ClinicalTrials.gov,146
1,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT03146338,Prophylaxis of Magnesium-rich Mineral Water to...,UNKNOWN,2017-07-04,2023-01-04,INTERVENTIONAL,NA,Metastatic Colorectal Cancer | Metastatic Head...,Magnesium-rich mineral water (Rozana) (OTHER),https://clinicaltrials.gov/study/NCT03146338,ClinicalTrials.gov,146
2,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT00346099,Study of Panitumumab Given First With Capecita...,WITHDRAWN,2006-06,2007-05,INTERVENTIONAL,PHASE2,Rectal Cancer | Neoplasm Metastasis,Panitumumab with capecitabine and oxaliplatin ...,https://clinicaltrials.gov/study/NCT00346099,ClinicalTrials.gov,146
3,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT01668498,Comparison of Two Preemptive Treatment Strateg...,COMPLETED,2011-05,2016-03,INTERVENTIONAL,PHASE2,Ras-wildtype Colorectal Cancer,Erythromycin (DRUG) | Doxycycline (DRUG),https://clinicaltrials.gov/study/NCT01668498,ClinicalTrials.gov,146
4,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT04034173,Optimal Anti-EGFR Treatment of mCRC Patients W...,NOT_YET_RECRUITING,2019-08-01,2026-08-01,INTERVENTIONAL,PHASE2,Treatment Related Cancer,Panitumumab (DRUG) | Irinotecan (DRUG) | Folin...,https://clinicaltrials.gov/study/NCT04034173,ClinicalTrials.gov,146
5,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT04787341,PAnitumumab REchallenge Followed by REgorafeni...,ACTIVE_NOT_RECRUITING,2020-12-15,2026-06-30,INTERVENTIONAL,PHASE2,Colorectal Cancer,Regorafenib (DRUG) | Panitumumab (DRUG),https://clinicaltrials.gov/study/NCT04787341,ClinicalTrials.gov,146
6,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT01380262,Pre-emptive Low-dose Doxycycline During Anti-E...,UNKNOWN,2010-06,2011-09,OBSERVATIONAL,NaN,Colorectal Cancer | Skin Toxicities,,https://clinicaltrials.gov/study/NCT01380262,ClinicalTrials.gov,146
7,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT03986541,"AREG, EREG and EGFR: Response to Anti-EGFR Age...",COMPLETED,2019-09-23,2022-07-22,OBSERVATIONAL,NaN,Colorectal Cancer Stage IV,Immunohistochemistry (DIAGNOSTIC_TEST),https://clinicaltrials.gov/study/NCT03986541,ClinicalTrials.gov,146
8,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT00115765,PACCE: Panitumumab Advanced Colorectal Cancer ...,COMPLETED,2005-06-01,2009-05-01,INTERVENTIONAL,PHASE3,Colorectal Cancer,Oxaliplatin Based Chemotherapy (DRUG) | Panitu...,https://clinicaltrials.gov/study/NCT00115765,ClinicalTrials.gov,146
9,EGFR,PANITUMUMAB,PANITUMUMAB EGFR,NCT01393821,Menadione Topical Lotion in Treating Skin Disc...,COMPLETED,2012-01-23,2018-07-14,INTERVENTIONAL,NA,Dermatologic Complications | Malignant Neoplas...,menadione topical lotion (DRUG) | placebo (OTH...,https://clinicaltrials.gov/study/NCT01393821,ClinicalTrials.gov,146
